In [16]:
import pandas as pd

# KCGS
kcgs = pd.read_csv('../data/kcgs.csv')

# 기업 단위 GIR
gir = pd.read_csv('../data/gir_company_2020_2025.csv')

print('KCGS:', kcgs.shape)
print('GIR:', gir.shape)

# 분석기간 통일
kcgs_merge = kcgs[
    kcgs['평가년도'].between(2020, 2025)
].copy()

gir_merge = gir[
    gir['대상년도'].between(2020, 2025)
].copy()

print('KCGS 2020~2025:', kcgs_merge.shape)
print('GIR 2020~2025:', gir_merge.shape)

KCGS: (12081, 13)
GIR: (6713, 11)
KCGS 2020~2025: (6203, 13)
GIR 2020~2025: (6713, 11)


In [17]:
# 기업명 앞뒤 공백 정리
kcgs_merge['기업명_clean'] = (
    kcgs_merge['기업명']
    .astype(str)
    .str.strip()
)

gir_merge['기업명_clean'] = (
    gir_merge['법인명']
    .astype(str)
    .str.strip()
)

# 같은 연도 + 같은 기업명이 중복되는지 확인
kcgs_name_dup = kcgs_merge.duplicated(
    subset=['기업명_clean', '평가년도'],
    keep=False
).sum()

gir_name_dup = gir_merge.duplicated(
    subset=['기업명_clean', '대상년도'],
    keep=False
).sum()

print('KCGS 기업명+연도 중복 행:', kcgs_name_dup)
print('GIR 기업명+연도 중복 행:', gir_name_dup)

KCGS 기업명+연도 중복 행: 0
GIR 기업명+연도 중복 행: 0


In [18]:
# 기업명 + 연도가 정확히 같은 경우만 우선 병합
exact_match = pd.merge(
    kcgs_merge,
    gir_merge,
    left_on=['기업명_clean', '평가년도'],
    right_on=['기업명_clean', '대상년도'],
    how='inner'
)

print('정확 일치 기업-연도 수:', len(exact_match))
print('정확 일치 기업 수:', exact_match['기업명_clean'].nunique())

# 연도별 정확 일치 건수
exact_match_year = (
    exact_match
    .groupby('평가년도')
    .size()
)

exact_match_year

정확 일치 기업-연도 수: 106
정확 일치 기업 수: 21


평가년도
2020.0    19
2021.0    20
2022.0    18
2023.0    17
2024.0    16
2025.0    16
dtype: int64

In [19]:
exact_match_rate = (
    len(exact_match)
    / len(kcgs_merge)
    * 100
)

print(
    'KCGS 기준 정확 일치율:',
    round(exact_match_rate, 1),
    '%'
)

KCGS 기준 정확 일치율: 1.7 %


In [20]:
# 정확 일치된 기업-연도 키
matched_kcgs_keys = set(
    zip(
        exact_match['기업명_clean'],
        exact_match['평가년도']
    )
)

matched_gir_keys = set(
    zip(
        exact_match['기업명_clean'],
        exact_match['대상년도']
    )
)

# 정확 일치하지 않은 KCGS
kcgs_unmatched = kcgs_merge[
    ~kcgs_merge.apply(
        lambda x: (
            x['기업명_clean'],
            x['평가년도']
        ) in matched_kcgs_keys,
        axis=1
    )
].copy()

# 정확 일치하지 않은 GIR
gir_unmatched = gir_merge[
    ~gir_merge.apply(
        lambda x: (
            x['기업명_clean'],
            x['대상년도']
        ) in matched_gir_keys,
        axis=1
    )
].copy()

print('KCGS 미매칭:', len(kcgs_unmatched))
print('GIR 미매칭:', len(gir_unmatched))

KCGS 미매칭: 6097
GIR 미매칭: 6607


In [21]:
import re

def normalize_company_name(name):

    name = str(name).strip()

    # 흔한 법인 표기 제거
    remove_words = [
        '주식회사',
        '(주)',
        '㈜',
        '유한회사',
        '(유)'
    ]

    for word in remove_words:
        name = name.replace(word, '')

    # 모든 공백 제거
    name = re.sub(r'\s+', '', name)

    return name


kcgs_unmatched['기업명_norm'] = (
    kcgs_unmatched['기업명_clean']
    .apply(normalize_company_name)
)

gir_unmatched['기업명_norm'] = (
    gir_unmatched['기업명_clean']
    .apply(normalize_company_name)
)

In [22]:
# 정규화 후 같은 연도 + 같은 이름 중복 확인
kcgs_norm_dup = kcgs_unmatched.duplicated(
    subset=['기업명_norm', '평가년도'],
    keep=False
).sum()

gir_norm_dup = gir_unmatched.duplicated(
    subset=['기업명_norm', '대상년도'],
    keep=False
).sum()

print('KCGS 정규화 후 중복 행:', kcgs_norm_dup)
print('GIR 정규화 후 중복 행:', gir_norm_dup)

KCGS 정규화 후 중복 행: 0
GIR 정규화 후 중복 행: 0


In [23]:
# 정규화 후 중복이 없는 경우에만 2차 매칭
if kcgs_norm_dup == 0 and gir_norm_dup == 0:

    second_match = pd.merge(
        kcgs_unmatched,
        gir_unmatched,
        left_on=['기업명_norm', '평가년도'],
        right_on=['기업명_norm', '대상년도'],
        how='inner',
        suffixes=('_kcgs', '_gir')
    )

    print('2차 추가 매칭 기업-연도 수:', len(second_match))
    print('2차 추가 매칭 기업 수:', second_match['기업명_norm'].nunique())

    print('\n연도별 추가 매칭 수')
    print(
        second_match
        .groupby('평가년도')
        .size()
    )

else:
    print('정규화 후 중복이 있어서 먼저 확인해야 합니다.')

2차 추가 매칭 기업-연도 수: 1299
2차 추가 매칭 기업 수: 277

연도별 추가 매칭 수
평가년도
2020.0    204
2021.0    201
2022.0    213
2023.0    222
2024.0    228
2025.0    231
dtype: int64


In [24]:
# 전체 매칭 건수
total_matched = len(exact_match) + len(second_match)

total_match_rate = (
    total_matched
    / len(kcgs_merge)
    * 100
)

print('1차 정확 매칭:', len(exact_match))
print('2차 추가 매칭:', len(second_match))
print('전체 매칭:', total_matched)

print(
    'KCGS 기준 전체 매칭률:',
    round(total_match_rate, 1),
    '%'
)

1차 정확 매칭: 106
2차 추가 매칭: 1299
전체 매칭: 1405
KCGS 기준 전체 매칭률: 22.7 %


In [25]:
# 2차 매칭 중 실제 원래 기업명이 달랐던 경우 확인
second_match_check = second_match[
    second_match['기업명_clean_kcgs']
    != second_match['기업명_clean_gir']
].copy()

print(
    '원래 표기가 달랐던 2차 매칭 수:',
    len(second_match_check)
)

# 앞부분 확인
second_match_check[
    [
        '기업명_clean_kcgs',
        '기업명_clean_gir',
        '기업명_norm',
        '평가년도',
        '환경',
        '온실가스 배출량(tCO₂eq)'
    ]
].head(30)

원래 표기가 달랐던 2차 매칭 수: 1299


,기업명_clean_kcgs,기업명_clean_gir,기업명_norm,평가년도,환경,온실가스 배출량(tCO₂eq)
0,CJ대한통운,CJ대한통운(주),CJ대한통운,2020.0,A,227703.0
1,CJ대한통운,CJ대한통운(주),CJ대한통운,2021.0,A,228274.0
2,CJ대한통운,CJ대한통운(주),CJ대한통운,2022.0,B+,228492.0
3,CJ대한통운,CJ대한통운(주),CJ대한통운,2023.0,A,227513.0
4,CJ대한통운,CJ대한통운(주),CJ대한통운,2024.0,A,228750.0
5,CJ대한통운,CJ대한통운(주),CJ대한통운,2025.0,A,239997.0
6,CJ제일제당,CJ제일제당(주),CJ제일제당,2020.0,A,405352.0
7,CJ제일제당,CJ제일제당(주),CJ제일제당,2021.0,A,426590.0
8,CJ제일제당,CJ제일제당(주),CJ제일제당,2022.0,A,427319.0
9,CJ제일제당,CJ제일제당(주),CJ제일제당,2023.0,A,411439.0


In [26]:
# 2차 매칭된 기업명+연도 키
second_kcgs_keys = set(
    zip(
        second_match['기업명_norm'],
        second_match['평가년도']
    )
)

second_gir_keys = set(
    zip(
        second_match['기업명_norm'],
        second_match['대상년도']
    )
)

# 2차까지 매칭되지 않은 KCGS
kcgs_remaining = kcgs_unmatched[
    ~kcgs_unmatched.apply(
        lambda x: (
            x['기업명_norm'],
            x['평가년도']
        ) in second_kcgs_keys,
        axis=1
    )
].copy()

# 2차까지 매칭되지 않은 GIR
gir_remaining = gir_unmatched[
    ~gir_unmatched.apply(
        lambda x: (
            x['기업명_norm'],
            x['대상년도']
        ) in second_gir_keys,
        axis=1
    )
].copy()

print('2차 이후 KCGS 미매칭:', len(kcgs_remaining))
print('2차 이후 GIR 미매칭:', len(gir_remaining))

print('\nKCGS 미매칭 고유 기업 수:',
      kcgs_remaining['기업명_clean'].nunique())

print('GIR 미매칭 고유 기업 수:',
      gir_remaining['기업명_clean'].nunique())

2차 이후 KCGS 미매칭: 4798
2차 이후 GIR 미매칭: 5308

KCGS 미매칭 고유 기업 수: 1146
GIR 미매칭 고유 기업 수: 1289


In [27]:
# KCGS 미매칭 기업 예시
kcgs_remaining[
    [
        '기업명_clean',
        '평가년도',
        '환경'
    ]
].head(30)

,기업명_clean,평가년도,환경
4,AJ네트웍스,2020.0,D
5,AJ네트웍스,2021.0,D
6,AJ네트웍스,2022.0,C
7,AJ네트웍스,2023.0,C
8,AJ네트웍스,2024.0,C
9,AJ네트웍스,2025.0,C
24,AK홀딩스,2020.0,C
25,AK홀딩스,2021.0,B
26,AK홀딩스,2022.0,B
27,AK홀딩스,2023.0,A


In [29]:
# 전체 데이터에도 동일한 기업명 정규화 적용
kcgs_merge['기업명_norm'] = (
    kcgs_merge['기업명_clean']
    .apply(normalize_company_name)
)

gir_merge['기업명_norm'] = (
    gir_merge['기업명_clean']
    .apply(normalize_company_name)
)

# KCGS + GIR 최종 병합
kg_merge = pd.merge(
    kcgs_merge,
    gir_merge,
    left_on=['기업명_norm', '평가년도'],
    right_on=['기업명_norm', '대상년도'],
    how='inner',
    suffixes=('_kcgs', '_gir'),
    validate='one_to_one'
)

print('최종 기업-연도 수:', len(kg_merge))
print('최종 고유 기업 수:', kg_merge['기업명_norm'].nunique())

print('\n연도별 매칭 수')
print(
    kg_merge
    .groupby('평가년도')
    .size()
)

최종 기업-연도 수: 1405
최종 고유 기업 수: 295

연도별 매칭 수
평가년도
2020.0    223
2021.0    221
2022.0    231
2023.0    239
2024.0    244
2025.0    247
dtype: int64


In [30]:
# 환경등급 분포 확인
print(
    kg_merge['환경']
    .value_counts(dropna=False)
)

환경
C       395
A       334
B+      304
B       244
D        52
A+       46
등급없음     30
Name: count, dtype: int64


In [31]:
# 실제 환경등급만 사용
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']

# 분석에 필요한 등급만 추출
kg_env = kg_merge[
    kg_merge['환경'].isin(grade_order)
].copy()

print('분석 대상 기업-연도:', len(kg_env))

# 환경등급별 온실가스 배출량 기본 통계
env_ghg_summary = (
    kg_env
    .groupby('환경')
    .agg(
        기업수=('기업명_norm', 'count'),
        총배출량=('온실가스 배출량(tCO₂eq)', 'sum'),
        평균배출량=('온실가스 배출량(tCO₂eq)', 'mean'),
        중앙값배출량=('온실가스 배출량(tCO₂eq)', 'median')
    )
    .reindex(grade_order)
    .round(1)
)

env_ghg_summary


분석 대상 기업-연도: 1375


,기업수,총배출량,평균배출량,중앙값배출량
환경,,,,
A+,46,44947366.0,977116.7,158614.5
A,334,394413540.0,1180878.9,152580.5
B+,304,305826259.0,1006007.4,101392.5
B,244,80317883.0,329171.7,62480.0
C,395,48824641.0,123606.7,50727.0
D,52,6582034.0,126577.6,32602.5


### 환경등급별 온실가스 배출량

KCGS 환경등급과 GIR 온실가스 배출량을 같은 기업·같은 연도 기준으로 결합한 결과, 실제 환경등급이 존재하는 1,375개 기업-연도 데이터를 확보했다.

환경등급별 절대 온실가스 배출량을 비교하면 A+, A, B+ 등급 기업의 중앙값 배출량이 B, C, D 등급보다 높게 나타났다.

하지만 절대 배출량은 기업 규모와 업종의 영향을 크게 받기 때문에, 이를 환경등급이 높은 기업의 환경성과가 낮다는 의미로 해석할 수는 없다.

또한 2022년에는 KCGS 환경등급 분포가 크게 변한 시기였으며 D등급 표본도 2022년에 집중되어 있어, 전체 기간을 단순 합산한 결과에는 연도별 등급 구성 차이가 영향을 미쳤을 가능성이 있다.

따라서 다음 분석에서는 연도별로 환경등급과 배출량의 관계를 다시 확인하고, 이후 DART 매출액을 결합해 기업 규모를 고려한 온실가스 배출집약도를 비교한다.

In [32]:
# 연도별 · 환경등급별 온실가스 배출량 중앙값
env_year_median = (
    kg_env
    .pivot_table(
        index='평가년도',
        columns='환경',
        values='온실가스 배출량(tCO₂eq)',
        aggfunc='median'
    )
    .reindex(columns=grade_order)
    .round(1)
)

env_year_median

환경,A+,A,B+,B,C,D
평가년도,,,,,,
2020.0,138131.0,298159.5,125538.5,52602.0,51626.0,578764.5
2021.0,901480.0,158747.0,94342.0,63161.0,55041.0,1201539.0
2022.0,541669.0,157431.5,178541.0,141177.0,56838.0,36675.0
2023.0,108517.0,215255.0,77855.0,100536.0,50724.0,26931.5
2024.0,217123.0,115544.5,78106.5,52382.5,36535.5,58403.0
2025.0,130937.0,123522.0,63510.0,89198.0,35794.0,32838.0


In [34]:
from scipy.stats import spearmanr

# 환경등급 순서만 숫자로 변환
grade_rank = {
    'D': 1,
    'C': 2,
    'B': 3,
    'B+': 4,
    'A': 5,
    'A+': 6
}

kg_env['환경등급순위'] = (
    kg_env['환경']
    .map(grade_rank)
)

year_corr = []

for year in sorted(kg_env['평가년도'].unique()):

    temp = kg_env[
        kg_env['평가년도'] == year
    ].dropna(
        subset=[
            '환경등급순위',
            '온실가스 배출량(tCO₂eq)'
        ]
    )

    corr, p_value = spearmanr(
        temp['환경등급순위'],
        temp['온실가스 배출량(tCO₂eq)']
    )

    year_corr.append({
        '연도': year,
        '기업수': len(temp),
        'Spearman상관계수': round(corr, 3),
        'p값': round(p_value, 4)
    })

year_corr = pd.DataFrame(year_corr)

year_corr

,연도,기업수,Spearman상관계수,p값
0,2020.0,218,0.350,0.0
1,2021.0,216,0.285,0.0
2,2022.0,226,0.414,0.0
3,2023.0,234,0.387,0.0
4,2024.0,239,0.379,0.0
5,2025.0,242,0.373,0.0


In [35]:
# DART 손익계산서 데이터 불러오기

dart_pl_list = []

for year in range(2020, 2026):

    df = pd.read_csv(
        f'../data/dart_pl_{year}.csv',
        low_memory=False
    )

    print(f'\n[{year}]')
    print('크기:', df.shape)
    print('컬럼:')
    print(df.columns.tolist())

    dart_pl_list.append(df)


[2020]
크기: (108501, 20)
컬럼:
['source_file', '재무제표종류', '종목코드', '회사명', '시장구분', '업종', '업종명', '결산월', '결산기준일', '보고서종류', '통화', '항목코드', '항목명', 'Unnamed: 12', '당기', 'Unnamed: 14', 'Unnamed: 15', '전기', '전전기', 'Unnamed: 18']

[2021]
크기: (111926, 20)
컬럼:
['source_file', '재무제표종류', '종목코드', '회사명', '시장구분', '업종', '업종명', '결산월', '결산기준일', '보고서종류', '통화', '항목코드', '항목명', 'Unnamed: 12', '당기', 'Unnamed: 14', 'Unnamed: 15', '전기', '전전기', 'Unnamed: 18']

[2022]
크기: (114802, 20)
컬럼:
['source_file', '재무제표종류', '종목코드', '회사명', '시장구분', '업종', '업종명', '결산월', '결산기준일', '보고서종류', '통화', '항목코드', '항목명', 'Unnamed: 12', '당기', 'Unnamed: 14', 'Unnamed: 15', '전기', '전전기', 'Unnamed: 18']

[2023]
크기: (122851, 20)
컬럼:
['source_file', '재무제표종류', '종목코드', '회사명', '시장구분', '업종', '업종명', '결산월', '결산기준일', '보고서종류', '통화', '항목코드', '항목명', 'Unnamed: 12', '당기', 'Unnamed: 14', 'Unnamed: 15', '전기', '전전기', 'Unnamed: 18']

[2024]
크기: (124412, 20)
컬럼:
['source_file', '재무제표종류', '종목코드', '회사명', '시장구분', '업종', '업종명', '결산월', '결산기준일', '보고서종류', '통화', '항목코드', '항목명',

In [36]:
dart_pl_list[-1].head()

,source_file,재무제표종류,종목코드,회사명,시장구분,업종,업종명,결산월,결산기준일,보고서종류,통화,항목코드,항목명,Unnamed: 12,당기,Unnamed: 14,Unnamed: 15,전기,전전기,Unnamed: 18
0,2025_사업보고서_02_손익계산서_20260902.csv,"손익계산서, 성격별 분류 - 별도",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2025-12-31,사업보고서,KRW,dart_NonOperatingProfitLoss,기타순손익,NaN,"1,527,951,000",NaN,NaN,"160,081,113,000","113,188,000",NaN
1,2025_사업보고서_02_손익계산서_20260902.csv,"손익계산서, 성격별 분류 - 별도",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2025-12-31,사업보고서,KRW,dart_OperatingIncomeLoss,영업이익,NaN,"146,483,268,000",NaN,NaN,"136,262,575,000","132,419,635,000",NaN
2,2025_사업보고서_02_손익계산서_20260902.csv,"손익계산서, 성격별 분류 - 별도",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2025-12-31,사업보고서,KRW,dart_BasicEarningsLossPerSharePreferredStock,우선주 기본주당이익,NaN,"4,088",NaN,NaN,"6,767","3,353",NaN
3,2025_사업보고서_02_손익계산서_20260902.csv,"손익계산서, 성격별 분류 - 별도",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2025-12-31,사업보고서,KRW,dart_DilutedEarningsLossPerSharePreferredStock,우선주 희석주당이익,NaN,"4,088",NaN,NaN,"6,767","3,353",NaN
4,2025_사업보고서_02_손익계산서_20260902.csv,"손익계산서, 성격별 분류 - 별도",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2025-12-31,사업보고서,KRW,entity00148540_BasicEarningsLossPerShareOfNewP...,신형우선주 기본주당이익,NaN,"4,038",NaN,NaN,"6,717","3,303",NaN


In [37]:
# DART 손익계산서에서 필요한 컬럼 후보 확인

sample_pl = dart_pl_list[0]

keywords = [
    '회사',
    '종목',
    '코드',
    '재무',
    '연결',
    '항목',
    '계정',
    '당기',
    '금액',
    '매출'
]

for col in sample_pl.columns:
    if any(keyword in str(col) for keyword in keywords):
        print(col)

재무제표종류
종목코드
회사명
항목코드
항목명
당기


In [38]:
# 앞부분 확인
sample_pl.head(10)

,source_file,재무제표종류,종목코드,회사명,시장구분,업종,업종명,결산월,결산기준일,보고서종류,통화,항목코드,항목명,Unnamed: 12,당기,Unnamed: 14,Unnamed: 15,전기,전전기,Unnamed: 18
0,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,entity00148540_udf_IS_2021317164541718_IncomeS...,영업수익,NaN,"165,163,104,000",NaN,NaN,"181,742,606,000","169,945,417,000",NaN
1,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,dart_TotalSellingGeneralAdministrativeExpenses,판매비와관리비,NaN,"75,141,496,000",NaN,NaN,"107,006,676,000","90,982,307,000",NaN
2,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,dart_OperatingIncomeLoss,영업이익,NaN,"90,021,608,000",NaN,NaN,"74,735,930,000","78,963,110,000",NaN
3,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_GainsArisingFromDerecognitionOfFinan...,기타순손익,NaN,"-20,421,085,000",NaN,NaN,"-468,945,000","-54,615,930,000",NaN
4,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_FinanceIncome,금융수익,NaN,"3,320,184,000",NaN,NaN,"5,013,004,000","20,400,895,000",NaN
5,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_FinanceCosts,금융비용,NaN,"6,342,583,000",NaN,NaN,"11,600,895,000","4,319,173,000",NaN
6,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_ProfitLossBeforeTax,법인세비용차감전순이익,NaN,"66,578,124,000",NaN,NaN,"67,679,094,000","40,428,902,000",NaN
7,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_IncomeTaxExpenseContinuingOperations,법인세비용,NaN,"14,391,342,000",NaN,NaN,"4,397,201,000","4,283,855,000",NaN
8,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_ProfitLoss,당기순이익,NaN,"52,186,782,000",NaN,NaN,"63,281,893,000","36,145,047,000",NaN
9,2020_사업보고서_02_손익계산서_20260403.csv,"손익계산서, 기능별 분류 - 별도재무제표",[001040],CJ,유가증권시장상장법인,649,기타 금융업,12,2020-12-31,사업보고서,KRW,ifrs-full_EarningsPerShareAbstract,주당이익 [abstract],NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# '매출'이라는 값이 들어있는 컬럼과 행 찾기

for col in sample_pl.columns:

    temp = sample_pl[col].astype(str)

    mask = temp.str.contains(
        '매출',
        na=False
    )

    if mask.any():

        print('\n컬럼:', col)
        print(
            sample_pl.loc[
                mask,
                [col]
            ].drop_duplicates().head(20)
        )


컬럼: 항목명
                               항목명
12                             매출액
13                            매출원가
14                           매출총이익
26                         수익(매출액)
109                           공사매출
110                           분양매출
111                           기타매출
113                         공사매출원가
114                         분양매출원가
115                         기타매출원가
130                          상품매출액
131                          수수료매출
133                         상품매출원가
134                        수수료매출원가
193                          제품매출액
196                         제품매출원가
215              건설계약으로 인한 수익(매출액)
217                       분양매출(수익)
218                      기타수익(매출액)
220     건설계약으로 인한 수익(매출액)에 대한 매출원가


In [40]:
# DART 손익계산서 컬럼 구조 확인

sample_pl = dart_pl_list[0]

print('전체 컬럼')
for i, col in enumerate(sample_pl.columns):
    print(i, col)

print('\n매출 관련 값이 들어있는 컬럼 확인')

for col in sample_pl.columns:

    text_col = sample_pl[col].astype(str)

    mask = text_col.str.contains(
        '매출|영업수익|수익',
        case=False,
        na=False
    )

    if mask.any():

        print(f'\n[{col}]')

        print(
            sample_pl.loc[
                mask,
                [col]
            ]
            .drop_duplicates()
            .head(30)
        )

전체 컬럼
0 source_file
1 재무제표종류
2 종목코드
3 회사명
4 시장구분
5 업종
6 업종명
7 결산월
8 결산기준일
9 보고서종류
10 통화
11 항목코드
12 항목명
13 Unnamed: 12
14 당기
15 Unnamed: 14
16 Unnamed: 15
17 전기
18 전전기
19 Unnamed: 18

매출 관련 값이 들어있는 컬럼 확인

[항목명]
                               항목명
0                             영업수익
4                             금융수익
12                             매출액
13                            매출원가
14                           매출총이익
19                            기타수익
26                         수익(매출액)
48                         기타영업외수익
109                           공사매출
110                           분양매출
111                           기타매출
113                         공사매출원가
114                         분양매출원가
115                         기타매출원가
130                          상품매출액
131                          수수료매출
133                         상품매출원가
134                        수수료매출원가
176                          영업외수익
179                           이자수익
193                          제품매출액
196                 

In [41]:
# 연결 / 별도 관련 값 확인

for col in sample_pl.columns:

    text_col = sample_pl[col].astype(str)

    mask = text_col.str.contains(
        '연결|별도',
        na=False
    )

    if mask.any():

        print(f'\n[{col}]')

        print(
            sample_pl.loc[
                mask,
                [col]
            ]
            .drop_duplicates()
            .head(20)
        )


[source_file]
                                 source_file
3529     2020_사업보고서_02_손익계산서_연결_20260403.csv
55248  2020_사업보고서_03_포괄손익계산서_연결_20260403.csv

[재무제표종류]
                                       재무제표종류
0                      손익계산서, 기능별 분류 - 별도재무제표
316                    손익계산서, 성격별 분류 - 별도재무제표
3529                   손익계산서, 기능별 분류 - 연결재무제표
4138                   손익계산서, 성격별 분류 - 연결재무제표
7439       포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표
7497   포괄손익계산서, 기능별 분류(세전기타포괄손익) - 별도재무제표(선택)
7689               포괄손익계산서(세후기타포괄손익) - 별도재무제표
8381           포괄손익계산서(세전기타포괄손익) - 별도재무제표(선택)
9261       포괄손익계산서, 성격별 분류(세후기타포괄손익) - 별도재무제표
13240  포괄손익계산서, 성격별 분류(세전기타포괄손익) - 별도재무제표(선택)
55248      포괄손익계산서, 기능별 분류(세후기타포괄손익) - 연결재무제표
55348  포괄손익계산서, 기능별 분류(세전기타포괄손익) - 연결재무제표(선택)
55522              포괄손익계산서(세후기타포괄손익) - 연결재무제표
56482          포괄손익계산서(세전기타포괄손익) - 연결재무제표(선택)
57532      포괄손익계산서, 성격별 분류(세후기타포괄손익) - 연결재무제표
62426  포괄손익계산서, 성격별 분류(세전기타포괄손익) - 연결재무제표(선택)

[항목명]
                       항목명
4851               연결당기순

In [42]:
import re
import pandas as pd

def clean_stock_code(x):
    # [005930], 005930, 5930 같은 값을 모두 6자리로 정리
    code = re.sub(r'[^0-9]', '', str(x))

    if code == '':
        return None

    return code.zfill(6)

In [43]:
def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    return None


dart_revenue_list = []

for year, df in zip(range(2020, 2026), dart_pl_list):

    temp = df.copy()

    # 주요 컬럼 찾기
    code_col = find_column(
        temp,
        ['종목코드', '회사코드']
    )

    company_col = find_column(
        temp,
        ['회사명', '기업명']
    )

    account_col = find_column(
        temp,
        ['항목명', '계정명', 'XBRL계정명']
    )

    fs_col = find_column(
        temp,
        ['재무제표종류', '재무제표 종류', '재무제표구분']
    )

    value_col = find_column(
        temp,
        ['당기', '당기금액', '당기(당분기,당반기)']
    )

    print(f'\n[{year}]')
    print('종목코드:', code_col)
    print('회사명:', company_col)
    print('계정명:', account_col)
    print('재무제표구분:', fs_col)
    print('당기값:', value_col)


[2020]
종목코드: 종목코드
회사명: 회사명
계정명: 항목명
재무제표구분: 재무제표종류
당기값: 당기

[2021]
종목코드: 종목코드
회사명: 회사명
계정명: 항목명
재무제표구분: 재무제표종류
당기값: 당기

[2022]
종목코드: 종목코드
회사명: 회사명
계정명: 항목명
재무제표구분: 재무제표종류
당기값: 당기

[2023]
종목코드: 종목코드
회사명: 회사명
계정명: 항목명
재무제표구분: 재무제표종류
당기값: 당기

[2024]
종목코드: 종목코드
회사명: 회사명
계정명: 항목명
재무제표구분: 재무제표종류
당기값: 당기

[2025]
종목코드: 종목코드
회사명: 회사명
계정명: 항목명
재무제표구분: 재무제표종류
당기값: 당기


In [44]:
# 2020~2025년 매출 관련 계정명 확인
revenue_account_list = []

for year, df in zip(range(2020, 2026), dart_pl_list):

    temp = df.copy()

    account_col = find_column(
        temp,
        ['항목명', '계정명', 'XBRL계정명']
    )

    if account_col is None:
        print(f'{year}: 계정명 컬럼을 찾지 못함')
        continue

    # 매출 / 수익 / 영업수익 관련 계정
    mask = (
        temp[account_col]
        .astype(str)
        .str.contains(
            '매출액|수익\\(매출액\\)|영업수익|매출',
            regex=True,
            na=False
        )
    )

    accounts = (
        temp.loc[mask, account_col]
        .value_counts()
        .reset_index()
    )

    accounts.columns = ['계정명', '건수']
    accounts['연도'] = year

    revenue_account_list.append(accounts)

# 전체 연도 합치기
revenue_accounts = pd.concat(
    revenue_account_list,
    ignore_index=True
)

# 많이 등장한 계정부터 확인
(
    revenue_accounts
    .groupby('계정명')['건수']
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

계정명
      매출원가                        12828
      매출총이익                       12484
매출원가                              11163
매출총이익                             11035
      매출액                          7001
매출액                                5965
      수익(매출액)                      5278
수익(매출액)                            4744
      영업수익                         1884
영업수익                               1135
      매출                            601
매출                                  496
   상품매출원가                           388
   제품매출원가                           376
      매출총이익(손실)                     359
         제품매출원가                     356
         상품매출원가                     343
   제품매출액                            301
   상품매출액                            300
         제품매출액                      277
   재화의 판매로 인한 수익(매출액)               276
         상품매출액                      270
   재화의 판매로 인한 수익(매출액)에 대한 매출원가      255
매출총이익(손실)                           244
      기타영업수익                        

In [45]:
# 연결 / 별도 구분 값 확인
for year, df in zip(range(2020, 2026), dart_pl_list):

    fs_col = find_column(
        df,
        ['재무제표종류', '재무제표 종류', '재무제표구분']
    )

    print(f'\n[{year}]')

    if fs_col is None:
        print('재무제표 구분 컬럼 없음')
    else:
        print(df[fs_col].value_counts(dropna=False).head(20))


[2020]
재무제표종류
포괄손익계산서, 기능별 분류(세후기타포괄손익) - 연결재무제표        47910
포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표        43119
손익계산서, 기능별 분류 - 연결재무제표                     3750
손익계산서, 기능별 분류 - 별도재무제표                     3351
포괄손익계산서, 기능별 분류(세전기타포괄손익) - 연결재무제표(선택)     2151
포괄손익계산서, 기능별 분류(세전기타포괄손익) - 별도재무제표(선택)     2013
포괄손익계산서(세후기타포괄손익) - 연결재무제표                 1967
포괄손익계산서(세후기타포괄손익) - 별도재무제표                 1327
포괄손익계산서, 성격별 분류(세후기타포괄손익) - 별도재무제표         1163
포괄손익계산서, 성격별 분류(세후기타포괄손익) - 연결재무제표         1011
손익계산서, 성격별 분류 - 별도재무제표                      178
손익계산서, 성격별 분류 - 연결재무제표                      160
포괄손익계산서(세전기타포괄손익) - 연결재무제표(선택)              130
포괄손익계산서, 성격별 분류(세전기타포괄손익) - 별도재무제표(선택)      109
포괄손익계산서, 성격별 분류(세전기타포괄손익) - 연결재무제표(선택)       84
포괄손익계산서(세전기타포괄손익) - 별도재무제표(선택)               78
Name: count, dtype: int64

[2021]
재무제표종류
포괄손익계산서, 기능별 분류(세후기타포괄손익) - 연결재무제표        49379
포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표        44236
손익계산서, 기능별 분류 - 연결재무제표                     3680
손익계산서, 기능별 분류 - 별도재무제표          

In [46]:
# 숫자형 변환 함수
def clean_number(x):
    return pd.to_numeric(
        str(x).replace(',', '').strip(),
        errors='coerce'
    )


revenue_candidates_list = []

# 매출액으로 인정할 계정명
revenue_accounts = [
    '매출액',
    '수익(매출액)',
    '영업수익'
]


for year, df in zip(range(2020, 2026), dart_pl_list):

    temp = df.copy()

    # 실제 컬럼명 찾기
    code_col = find_column(
        temp,
        ['종목코드', '회사코드']
    )

    company_col = find_column(
        temp,
        ['회사명', '기업명']
    )

    account_col = find_column(
        temp,
        ['항목명', '계정명', 'XBRL계정명']
    )

    fs_col = find_column(
        temp,
        ['재무제표종류', '재무제표 종류', '재무제표구분']
    )

    value_col = find_column(
        temp,
        [
            '당기',
            '당기금액',
            '당기(당분기,당반기)',
            '당기(당분기, 당반기)'
        ]
    )

    # 필요한 컬럼이 없으면 확인
    if None in [
        code_col,
        company_col,
        account_col,
        fs_col,
        value_col
    ]:
        print(year, '→ 필요한 컬럼 확인 필요')
        print(
            code_col,
            company_col,
            account_col,
            fs_col,
            value_col
        )
        continue

    # 계정명 앞뒤 공백 제거
    temp['계정명_clean'] = (
        temp[account_col]
        .astype(str)
        .str.strip()
    )

    # 매출 관련 계정만 추출
    temp = temp[
        temp['계정명_clean'].isin(
            revenue_accounts
        )
    ].copy()

    # 종목코드 정리
    temp['종목코드_clean'] = (
        temp[code_col]
        .apply(clean_stock_code)
    )

    # 매출액 숫자형 변환
    temp['매출액'] = (
        temp[value_col]
        .apply(clean_number)
    )

    # 연결 / 별도 구분
    temp['재무제표구분'] = temp[fs_col].astype(str)

    temp['연결여부'] = temp['재무제표구분'].apply(
        lambda x: '연결'
        if '연결' in x and '별도' not in x
        else '별도'
        if '별도' in x
        else '기타'
    )

    # 필요한 열만 저장
    result = temp[
        [
            '종목코드_clean',
            company_col,
            '계정명_clean',
            '매출액',
            '연결여부'
        ]
    ].copy()

    result = result.rename(
        columns={
            company_col: '회사명'
        }
    )

    result['연도'] = year

    revenue_candidates_list.append(result)

In [47]:
dart_revenue_candidates = pd.concat(
    revenue_candidates_list,
    ignore_index=True
)

print(
    '매출액 후보 전체 행:',
    len(dart_revenue_candidates)
)

print(
    '기업-연도 수:',
    dart_revenue_candidates[
        ['종목코드_clean', '연도']
    ].drop_duplicates().shape[0]
)

매출액 후보 전체 행: 26059
기업-연도 수: 13774


In [48]:
revenue_dup = (
    dart_revenue_candidates
    .groupby(
        ['종목코드_clean', '연도']
    )
    .size()
)

revenue_dup = revenue_dup[
    revenue_dup > 1
]

print(
    '매출액 후보가 2개 이상인 기업-연도:',
    len(revenue_dup)
)

revenue_dup.head(20)

매출액 후보가 2개 이상인 기업-연도: 11019


종목코드_clean  연도  
000010      2025    2
000020      2020    2
            2021    2
            2022    2
            2023    2
            2024    2
            2025    2
000040      2020    2
            2021    2
            2022    2
            2023    2
            2024    2
            2025    5
000050      2020    2
            2021    2
            2022    2
            2023    2
            2024    2
            2025    2
000070      2020    2
dtype: int64

In [50]:
import pandas as pd
import numpy as np
import re

# KCGS 종목코드 정리
def clean_kcgs_code(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    # CSV에서 120.0처럼 읽힌 경우 처리
    x = re.sub(r'\.0$', '', x)

    if x.isdigit():
        return x.zfill(6)

    return None


# DART 종목코드 정리
# 숫자 6자리 종목코드만 인정
def clean_dart_code(x):

    x = str(x).strip()

    # [005930] 형태
    match = re.fullmatch(r'\[(\d{6})\]', x)

    if match:
        return match.group(1)

    # 005930 형태
    if re.fullmatch(r'\d{6}', x):
        return x

    # 영문이 포함된 코드는 제외
    return None

In [51]:
dart_revenue_list = []

for year in range(2020, 2026):

    df = pd.read_csv(
        f'../data/dart_pl_{year}.csv',
        usecols=[
            '재무제표종류',
            '종목코드',
            '회사명',
            '결산기준일',
            '통화',
            '항목코드',
            '항목명',
            '당기'
        ],
        low_memory=False
    )

    # 실제 상장 종목코드만 정리
    df['종목코드_clean'] = (
        df['종목코드']
        .apply(clean_dart_code)
    )

    # 표준 Revenue 계정만 사용
    df = df[
        df['종목코드_clean'].notna()
        &
        (df['항목코드'] == 'ifrs-full_Revenue')
    ].copy()

    # 금액 숫자형 변환
    df['매출액'] = pd.to_numeric(
        df['당기']
        .astype(str)
        .str.replace(',', ''),
        errors='coerce'
    )

    # 결산일
    df['결산기준일_clean'] = pd.to_datetime(
        df['결산기준일'],
        errors='coerce'
    )

    # 연결 / 별도 구분
    df['연결여부'] = np.where(
        df['재무제표종류']
        .astype(str)
        .str.contains('연결'),
        '연결',
        '별도'
    )

    df['연도'] = year

    # 같은 기업·연도·재무제표 유형이 여러 개면
    # 해당 연도에서 가장 늦은 결산기준일 사용
    max_date = (
        df
        .groupby(
            ['종목코드_clean', '연도', '연결여부']
        )['결산기준일_clean']
        .transform('max')
    )

    df = df[
        df['결산기준일_clean'] == max_date
    ].copy()

    # 같은 결산일의 동일 값 중복 제거
    df = df.drop_duplicates(
        subset=[
            '종목코드_clean',
            '연도',
            '연결여부',
            '매출액'
        ]
    )

    dart_revenue_list.append(df)


dart_revenue = pd.concat(
    dart_revenue_list,
    ignore_index=True
)

In [52]:
# 별도재무제표를 우선 사용
dart_revenue['우선순위'] = (
    dart_revenue['연결여부']
    .map({
        '별도': 0,
        '연결': 1
    })
)

dart_revenue_final = (
    dart_revenue
    .sort_values(
        ['종목코드_clean', '연도', '우선순위']
    )
    .drop_duplicates(
        subset=['종목코드_clean', '연도'],
        keep='first'
    )
    .copy()
)

print(
    'DART 기업-연도 매출액:',
    len(dart_revenue_final)
)

print(
    dart_revenue_final[
        '연결여부'
    ].value_counts()
)

DART 기업-연도 매출액: 14381
연결여부
별도    14156
연결      225
Name: count, dtype: int64


In [53]:
# KCGS 기업코드 6자리 정리
kg_env['종목코드_clean'] = (
    kg_env['기업코드']
    .apply(clean_kcgs_code)
)

# KCGS + GIR + DART 결합
kg_fin = pd.merge(
    kg_env,
    dart_revenue_final[
        [
            '종목코드_clean',
            '연도',
            '매출액',
            '연결여부',
            '결산기준일_clean',
            '통화'
        ]
    ],
    left_on=[
        '종목코드_clean',
        '평가년도'
    ],
    right_on=[
        '종목코드_clean',
        '연도'
    ],
    how='left'
)

print('전체:', len(kg_fin))
print('매출액 매칭:', kg_fin['매출액'].notna().sum())

print(
    '매칭률:',
    round(
        kg_fin['매출액'].notna().mean() * 100,
        1
    ),
    '%'
)

전체: 1375
매출액 매칭: 1366
매칭률: 99.3 %


In [54]:
# 정상적인 매출액이 있는 데이터만 사용
intensity_df = kg_fin[
    (kg_fin['매출액'] > 0)
    &
    (kg_fin['통화'] == 'KRW')
].copy()

# 매출 10억원당 tCO2eq
intensity_df['배출집약도'] = (
    intensity_df['온실가스 배출량(tCO₂eq)']
    /
    (intensity_df['매출액'] / 1_000_000_000)
)

# 환경등급별 배출집약도
intensity_summary = (
    intensity_df
    .groupby('환경')
    .agg(
        기업연도=('기업명_norm', 'count'),
        평균집약도=('배출집약도', 'mean'),
        중앙값집약도=('배출집약도', 'median')
    )
    .reindex(
        ['A+', 'A', 'B+', 'B', 'C', 'D']
    )
    .round(1)
)

intensity_summary

,기업연도,평균집약도,중앙값집약도
환경,,,
A+,46,110.5,38.4
A,331,329.5,59.7
B+,304,580.8,87.9
B,243,446.9,125.1
C,391,383.7,171.6
D,51,287.1,151.9


In [56]:
# 배출집약도 분석 데이터 다시 만들기

# 매출액과 온실가스 배출량이 정상적으로 있는 데이터만 사용
intensity_df = kg_fin[
    (kg_fin['매출액'] > 0)
    &
    (kg_fin['통화'] == 'KRW')
].copy()

# 매출 10억원당 온실가스 배출량
intensity_df['배출집약도'] = (
    intensity_df['온실가스 배출량(tCO₂eq)']
    /
    (intensity_df['매출액'] / 1_000_000_000)
)

# 현재 데이터에서 '업종'이 들어간 컬럼 자동 확인
industry_columns = [
    col for col in intensity_df.columns
    if '업종' in str(col)
]

print('업종 관련 컬럼:', industry_columns)
print('분석 데이터 크기:', intensity_df.shape)

업종 관련 컬럼: ['지정업종(목표)/<br />계획업종(할당)']
분석 데이터 크기: (1366, 35)


In [57]:
# 실제 업종 컬럼 선택
industry_col = industry_columns[0]

# 업종별 표본 수
industry_count = (
    intensity_df[industry_col]
    .value_counts(dropna=False)
)

industry_count.head(20)

지정업종(목표)/<br />계획업종(할당)
산업                            179
석유화학                           91
1차 철강 제조업                      71
기초 화학물질 제조업                    67
전기전자                           65
펄프, 종이 및 판지 제조업                49
기계                             40
시멘트, 석회, 플라스터 및 그 외 제품 제조업     36
전자 부품 제조업                      34
철강                             32
자동차 신품 부품 제조업                  31
1차 비철금속 제조업                    29
비철금속                           27
기타 식품 제조업                      27
도로 화물 운송업                      25
건물                             24
자동차                            22
식료품 제조업                        21
건설                             20
반도체                            20
Name: count, dtype: int64

In [58]:
industry_col = '지정업종(목표)/계획업종(할당)'

In [59]:
# 업종별 표본 수와 환경등급 종류 확인

industry_col = industry_columns[0]

industry_check = (
    intensity_df
    .groupby(industry_col)
    .agg(
        기업연도수=('기업명_norm', 'size'),
        고유기업수=('기업명_norm', 'nunique'),
        환경등급종류=('환경', 'nunique')
    )
    .sort_values(
        '기업연도수',
        ascending=False
    )
)

industry_check.head(20)

,기업연도수,고유기업수,환경등급종류
지정업종(목표)/<br />계획업종(할당),,,
산업,179,149,6
석유화학,91,46,5
1차 철강 제조업,71,19,6
기초 화학물질 제조업,67,18,6
전기전자,65,24,6
"펄프, 종이 및 판지 제조업",49,14,5
기계,40,10,6
"시멘트, 석회, 플라스터 및 그 외 제품 제조업",36,10,5
전자 부품 제조업,34,9,5


In [60]:
# 표본이 어느 정도 확보된 업종만 선택
usable_industries = industry_check[
    (industry_check['기업연도수'] >= 30)
    &
    (industry_check['고유기업수'] >= 10)
    &
    (industry_check['환경등급종류'] >= 3)
].copy()

print('분석 가능한 업종 수:', len(usable_industries))

usable_industries

분석 가능한 업종 수: 9


,기업연도수,고유기업수,환경등급종류
지정업종(목표)/<br />계획업종(할당),,,
산업,179,149,6
석유화학,91,46,5
1차 철강 제조업,71,19,6
기초 화학물질 제조업,67,18,6
전기전자,65,24,6
"펄프, 종이 및 판지 제조업",49,14,5
기계,40,10,6
"시멘트, 석회, 플라스터 및 그 외 제품 제조업",36,10,5
철강,32,20,5


In [61]:
from scipy.stats import spearmanr
import pandas as pd

# 환경등급의 순서
grade_rank = {
    'D': 1,
    'C': 2,
    'B': 3,
    'B+': 4,
    'A': 5,
    'A+': 6
}

# 등급 순위 생성
intensity_df['환경등급순위'] = (
    intensity_df['환경']
    .map(grade_rank)
)

industry_corr_result = []

for industry in usable_industries.index:

    temp = intensity_df[
        intensity_df[industry_col] == industry
    ].dropna(
        subset=[
            '환경등급순위',
            '배출집약도'
        ]
    )

    # Spearman 상관계수
    corr, p_value = spearmanr(
        temp['환경등급순위'],
        temp['배출집약도']
    )

    industry_corr_result.append({
        '업종': industry,
        '기업연도수': len(temp),
        '고유기업수': temp['기업명_norm'].nunique(),
        '상관계수': round(corr, 3),
        'p값': round(p_value, 4)
    })

industry_corr = pd.DataFrame(
    industry_corr_result
)

# 환경등급이 높을수록 배출집약도가 낮은 업종부터
industry_corr = industry_corr.sort_values(
    '상관계수'
)

industry_corr

,업종,기업연도수,고유기업수,상관계수,p값
4,전기전자,65,24,-0.516,0.0000
6,기계,40,10,-0.446,0.0039
0,산업,179,149,-0.383,0.0000
2,1차 철강 제조업,71,19,-0.363,0.0019
3,기초 화학물질 제조업,67,18,-0.345,0.0042
1,석유화학,91,46,-0.321,0.0019
5,"펄프, 종이 및 판지 제조업",49,14,-0.306,0.0327
7,"시멘트, 석회, 플라스터 및 그 외 제품 제조업",36,10,0.085,0.6227
8,철강,32,20,0.455,0.0088


In [62]:
# 업종별 상관 방향 요약

negative_count = (
    industry_corr['상관계수'] < 0
).sum()

positive_count = (
    industry_corr['상관계수'] > 0
).sum()

zero_count = (
    industry_corr['상관계수'] == 0
).sum()

significant_negative = (
    (industry_corr['상관계수'] < 0)
    &
    (industry_corr['p값'] < 0.05)
).sum()

significant_positive = (
    (industry_corr['상관계수'] > 0)
    &
    (industry_corr['p값'] < 0.05)
).sum()


print('분석 업종 수:', len(industry_corr))

print('\n음(-)의 상관 업종:', negative_count)
print('양(+)의 상관 업종:', positive_count)
print('상관계수 0:', zero_count)

print('\np < 0.05인 음(-)의 상관:', significant_negative)
print('p < 0.05인 양(+)의 상관:', significant_positive)

분석 업종 수: 9

음(-)의 상관 업종: 7
양(+)의 상관 업종: 2
상관계수 0: 0

p < 0.05인 음(-)의 상관: 7
p < 0.05인 양(+)의 상관: 1


In [63]:
# 음의 관계가 강한 업종
print('음의 상관이 큰 업종')

display(
    industry_corr
    .sort_values('상관계수')
    .head(10)
)

# 양의 관계가 강한 업종
print('양의 상관이 큰 업종')

display(
    industry_corr
    .sort_values(
        '상관계수',
        ascending=False
    )
    .head(10)
)

음의 상관이 큰 업종


,업종,기업연도수,고유기업수,상관계수,p값
4,전기전자,65,24,-0.516,0.0000
6,기계,40,10,-0.446,0.0039
0,산업,179,149,-0.383,0.0000
2,1차 철강 제조업,71,19,-0.363,0.0019
3,기초 화학물질 제조업,67,18,-0.345,0.0042
1,석유화학,91,46,-0.321,0.0019
5,"펄프, 종이 및 판지 제조업",49,14,-0.306,0.0327
7,"시멘트, 석회, 플라스터 및 그 외 제품 제조업",36,10,0.085,0.6227
8,철강,32,20,0.455,0.0088


양의 상관이 큰 업종


,업종,기업연도수,고유기업수,상관계수,p값
8,철강,32,20,0.455,0.0088
7,"시멘트, 석회, 플라스터 및 그 외 제품 제조업",36,10,0.085,0.6227
5,"펄프, 종이 및 판지 제조업",49,14,-0.306,0.0327
1,석유화학,91,46,-0.321,0.0019
3,기초 화학물질 제조업,67,18,-0.345,0.0042
2,1차 철강 제조업,71,19,-0.363,0.0019
0,산업,179,149,-0.383,0.0000
6,기계,40,10,-0.446,0.0039
4,전기전자,65,24,-0.516,0.0000


In [64]:
# 같은 기업의 환경등급 변화와 온실가스 배출량 변화 비교

grade_rank = {
    'D': 1,
    'C': 2,
    'B': 3,
    'B+': 4,
    'A': 5,
    'A+': 6
}

change_df = kg_fin[
    kg_fin['환경'].isin(grade_rank.keys())
].copy()

# 환경등급을 순서형 숫자로 변환
change_df['환경등급순위'] = (
    change_df['환경']
    .map(grade_rank)
)

# 기업코드 + 연도 순서로 정렬
change_df = change_df.sort_values(
    ['종목코드_clean', '평가년도']
)

# 전년도 값 가져오기
change_df['전년도'] = (
    change_df
    .groupby('종목코드_clean')['평가년도']
    .shift(1)
)

change_df['전년환경등급순위'] = (
    change_df
    .groupby('종목코드_clean')['환경등급순위']
    .shift(1)
)

change_df['전년배출량'] = (
    change_df
    .groupby('종목코드_clean')['온실가스 배출량(tCO₂eq)']
    .shift(1)
)

# 실제 연속된 연도끼리만 비교
change_df = change_df[
    change_df['평가년도'] - change_df['전년도'] == 1
].copy()

# 환경등급 변화
change_df['등급변화'] = (
    change_df['환경등급순위']
    - change_df['전년환경등급순위']
)

change_df['등급변화구분'] = change_df['등급변화'].apply(
    lambda x:
    '상향' if x > 0
    else '하향' if x < 0
    else '유지'
)

# 전년도 배출량이 0인 경우 증감률 계산에서 제외
change_df['배출량증감률'] = (
    (
        change_df['온실가스 배출량(tCO₂eq)']
        - change_df['전년배출량']
    )
    /
    change_df['전년배출량']
    * 100
)

change_df.loc[
    change_df['전년배출량'] <= 0,
    '배출량증감률'
] = pd.NA

print('비교 가능한 기업-연도:', len(change_df))

print('\n환경등급 변화')
print(
    change_df['등급변화구분']
    .value_counts()
)

비교 가능한 기업-연도: 1083

환경등급 변화
등급변화구분
유지    583
상향    268
하향    232
Name: count, dtype: int64


In [65]:
# 환경등급 변화별 실제 배출량 변화

grade_change_summary = (
    change_df
    .groupby('등급변화구분')
    .agg(
        기업연도수=('종목코드_clean', 'size'),
        배출량증감률_평균=('배출량증감률', 'mean'),
        배출량증감률_중앙값=('배출량증감률', 'median'),
        배출감소기업수=(
            '배출량증감률',
            lambda x: (x < 0).sum()
        )
    )
)

grade_change_summary['배출감소비율'] = (
    grade_change_summary['배출감소기업수']
    /
    grade_change_summary['기업연도수']
    * 100
)

grade_change_summary.round(1)

,기업연도수,배출량증감률_평균,배출량증감률_중앙값,배출감소기업수,배출감소비율
등급변화구분,,,,,
상향,268,0.3,-0.6,143,53.4
유지,583,19.6,-0.3,302,51.8
하향,232,1.9,-0.3,117,50.4


In [66]:
# 비교 구간 만들기
change_df['비교기간'] = (
    change_df['전년도']
    .astype(int)
    .astype(str)
    + '→'
    + change_df['평가년도']
    .astype(int)
    .astype(str)
)

# 연도별 + 환경등급 변화별 배출량 변화
year_grade_change = (
    change_df
    .groupby(
        ['비교기간', '등급변화구분']
    )
    .agg(
        기업연도수=('종목코드_clean', 'size'),

        배출량증감률_중앙값=(
            '배출량증감률',
            'median'
        ),

        배출감소기업수=(
            '배출량증감률',
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

# 배출 감소 비율 계산
year_grade_change['배출감소비율'] = (
    year_grade_change['배출감소기업수']
    /
    year_grade_change['기업연도수']
    * 100
)

year_grade_change.round(1)

,비교기간,등급변화구분,기업연도수,배출량증감률_중앙값,배출감소기업수,배출감소비율
0,2020→2021,상향,47,3.0,18,38.3
1,2020→2021,유지,106,2.5,39,36.8
2,2020→2021,하향,49,2.4,16,32.7
3,2021→2022,상향,12,-1.2,7,58.3
4,2021→2022,유지,63,-0.1,32,50.8
5,2021→2022,하향,128,-1.2,70,54.7
6,2022→2023,상향,136,-1.5,80,58.8
7,2022→2023,유지,80,-3.6,51,63.7
8,2022→2023,하향,4,3.4,1,25.0
9,2023→2024,상향,42,-0.3,23,54.8


In [68]:
# 2022~2023의 큰 등급 변동 영향을 제외
# 2023→2024, 2024→2025만 사용

stable_change = change_df[
    change_df['평가년도'].isin([2024, 2025])
].copy()

stable_summary = (
    stable_change
    .groupby('등급변화구분')
    .agg(
        기업연도수=('종목코드_clean', 'size'),
        고유기업수=('종목코드_clean', 'nunique'),

        배출량증감률_중앙값=(
            '배출량증감률',
            'median'
        ),

        배출감소기업수=(
            '배출량증감률',
            lambda x: (x < 0).sum()
        )
    )
)

# 배출 감소 비율
stable_summary['배출감소비율'] = (
    stable_summary['배출감소기업수']
    /
    stable_summary['기업연도수']
    * 100
)

stable_summary.round(1)

,기업연도수,고유기업수,배출량증감률_중앙값,배출감소기업수,배출감소비율
등급변화구분,,,,,
상향,73,69,-0.2,38,52.1
유지,334,210,-0.6,180,53.9
하향,51,50,-1.5,30,58.8


In [69]:
# 환경등급 변화와 배출집약도 변화 비교

intensity_change = intensity_df[
    intensity_df['환경'].isin(grade_rank.keys())
].copy()

# 기업 + 연도 순으로 정렬
intensity_change = intensity_change.sort_values(
    ['종목코드_clean', '평가년도']
)

# 전년도 값 가져오기
intensity_change['전년도'] = (
    intensity_change
    .groupby('종목코드_clean')['평가년도']
    .shift(1)
)

intensity_change['전년환경등급순위'] = (
    intensity_change
    .groupby('종목코드_clean')['환경등급순위']
    .shift(1)
)

intensity_change['전년배출집약도'] = (
    intensity_change
    .groupby('종목코드_clean')['배출집약도']
    .shift(1)
)

# 연속된 연도만 비교
intensity_change = intensity_change[
    intensity_change['평가년도']
    - intensity_change['전년도'] == 1
].copy()

# 환경등급 변화
intensity_change['등급변화'] = (
    intensity_change['환경등급순위']
    - intensity_change['전년환경등급순위']
)

intensity_change['등급변화구분'] = (
    intensity_change['등급변화']
    .apply(
        lambda x:
        '상향' if x > 0
        else '하향' if x < 0
        else '유지'
    )
)

# 배출집약도 증감률
intensity_change['집약도증감률'] = (
    (
        intensity_change['배출집약도']
        - intensity_change['전년배출집약도']
    )
    /
    intensity_change['전년배출집약도']
    * 100
)

# 전년도 집약도가 0 이하인 경우 제외
intensity_change.loc[
    intensity_change['전년배출집약도'] <= 0,
    '집약도증감률'
] = pd.NA

In [70]:
intensity_change_summary = (
    intensity_change
    .groupby('등급변화구분')
    .agg(
        기업연도수=('종목코드_clean', 'size'),
        고유기업수=('종목코드_clean', 'nunique'),

        집약도증감률_중앙값=(
            '집약도증감률',
            'median'
        ),

        집약도감소기업수=(
            '집약도증감률',
            lambda x: (x < 0).sum()
        )
    )
)

intensity_change_summary['집약도감소비율'] = (
    intensity_change_summary['집약도감소기업수']
    /
    intensity_change_summary['기업연도수']
    * 100
)

intensity_change_summary.round(1)

,기업연도수,고유기업수,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
등급변화구분,,,,,
상향,266,179,-4.0,162,60.9
유지,571,241,-4.3,361,63.2
하향,230,180,-10.3,178,77.4


In [71]:
# 최근 안정 구간만 사용
recent_intensity_change = intensity_change[
    intensity_change['평가년도'].isin([2024, 2025])
].copy()

recent_intensity_summary = (
    recent_intensity_change
    .groupby('등급변화구분')
    .agg(
        기업연도수=('종목코드_clean', 'size'),
        고유기업수=('종목코드_clean', 'nunique'),

        집약도증감률_중앙값=(
            '집약도증감률',
            'median'
        ),

        집약도감소기업수=(
            '집약도증감률',
            lambda x: (x < 0).sum()
        )
    )
)

recent_intensity_summary['집약도감소비율'] = (
    recent_intensity_summary['집약도감소기업수']
    /
    recent_intensity_summary['기업연도수']
    * 100
)

recent_intensity_summary.round(1)

,기업연도수,고유기업수,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
등급변화구분,,,,,
상향,71,67,-4.0,46,64.8
유지,324,205,-2.3,194,59.9
하향,50,49,-2.0,30,60.0


In [72]:
# 2023→2024, 2024→2025 각각 확인

recent_intensity_change['비교기간'] = (
    recent_intensity_change['전년도']
    .astype(int)
    .astype(str)
    + '→'
    + recent_intensity_change['평가년도']
    .astype(int)
    .astype(str)
)

recent_year_summary = (
    recent_intensity_change
    .groupby(
        ['비교기간', '등급변화구분']
    )
    .agg(
        기업연도수=('종목코드_clean', 'size'),

        집약도증감률_중앙값=(
            '집약도증감률',
            'median'
        ),

        집약도감소기업수=(
            '집약도증감률',
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

recent_year_summary['집약도감소비율'] = (
    recent_year_summary['집약도감소기업수']
    /
    recent_year_summary['기업연도수']
    * 100
)

recent_year_summary.round(1)

,비교기간,등급변화구분,기업연도수,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
0,2023→2024,상향,40,-4.2,24,60.0
1,2023→2024,유지,142,-2.3,82,57.7
2,2023→2024,하향,36,-2.9,23,63.9
3,2024→2025,상향,31,-3.8,22,71.0
4,2024→2025,유지,182,-2.3,112,61.5
5,2024→2025,하향,14,-0.2,7,50.0


In [73]:
from scipy.stats import spearmanr

# 전체 기간
temp_all = intensity_change.dropna(
    subset=['등급변화', '집약도증감률']
)

corr_all, p_all = spearmanr(
    temp_all['등급변화'],
    temp_all['집약도증감률']
)

# 최근 안정 구간: 2023→2024, 2024→2025
temp_recent = recent_intensity_change.dropna(
    subset=['등급변화', '집약도증감률']
)

corr_recent, p_recent = spearmanr(
    temp_recent['등급변화'],
    temp_recent['집약도증감률']
)

print('전체 기간')
print('기업-연도:', len(temp_all))
print('Spearman 상관계수:', round(corr_all, 3))
print('p값:', round(p_all, 4))

print('\n2023→2025 안정 구간')
print('기업-연도:', len(temp_recent))
print('Spearman 상관계수:', round(corr_recent, 3))
print('p값:', round(p_recent, 4))

전체 기간
기업-연도: 1067
Spearman 상관계수: 0.164
p값: 0.0

2023→2025 안정 구간
기업-연도: 445
Spearman 상관계수: -0.087
p값: 0.0652


In [74]:
# KCGS + GIR + DART 최종 결과 확인

print('=== 1. 환경등급별 배출집약도 ===')
display(intensity_summary)

print('\n=== 2. 업종별 상관관계 ===')
print('분석 업종 수:', len(industry_corr))
print(
    '음(-)의 상관 업종:',
    (industry_corr['상관계수'] < 0).sum()
)
print(
    '양(+)의 상관 업종:',
    (industry_corr['상관계수'] > 0).sum()
)

print('\n=== 3. 최근 환경등급 변화와 절대배출량 변화 ===')
display(stable_summary.round(1))

print('\n=== 4. 최근 환경등급 변화와 배출집약도 변화 ===')
display(recent_intensity_summary.round(1))

print('\n=== 5. 환경등급 변화폭 vs 배출집약도 변화 ===')

print('전체 기간')
print('상관계수:', round(corr_all, 3))
print('p값:', round(p_all, 4))

print('\n2023→2025')
print('상관계수:', round(corr_recent, 3))
print('p값:', round(p_recent, 4))

=== 1. 환경등급별 배출집약도 ===


,기업연도,평균집약도,중앙값집약도
환경,,,
A+,46,110.5,38.4
A,331,329.5,59.7
B+,304,580.8,87.9
B,243,446.9,125.1
C,391,383.7,171.6
D,51,287.1,151.9



=== 2. 업종별 상관관계 ===
분석 업종 수: 9
음(-)의 상관 업종: 7
양(+)의 상관 업종: 2

=== 3. 최근 환경등급 변화와 절대배출량 변화 ===


,기업연도수,고유기업수,배출량증감률_중앙값,배출감소기업수,배출감소비율
등급변화구분,,,,,
상향,73,69,-0.2,38,52.1
유지,334,210,-0.6,180,53.9
하향,51,50,-1.5,30,58.8



=== 4. 최근 환경등급 변화와 배출집약도 변화 ===


,기업연도수,고유기업수,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
등급변화구분,,,,,
상향,71,67,-4.0,46,64.8
유지,324,205,-2.3,194,59.9
하향,50,49,-2.0,30,60.0



=== 5. 환경등급 변화폭 vs 배출집약도 변화 ===
전체 기간
상관계수: 0.164
p값: 0.0

2023→2025
상관계수: -0.087
p값: 0.0652


In [75]:
# KCGS + GIR + DART 최종 결과 확인

print('=== 1. 환경등급별 배출집약도 ===')
display(intensity_summary)

print('\n=== 2. 업종별 상관관계 ===')
print('분석 업종 수:', len(industry_corr))
print(
    '음(-)의 상관 업종:',
    (industry_corr['상관계수'] < 0).sum()
)
print(
    '양(+)의 상관 업종:',
    (industry_corr['상관계수'] > 0).sum()
)

print('\n=== 3. 최근 환경등급 변화와 절대배출량 변화 ===')
display(stable_summary.round(1))

print('\n=== 4. 최근 환경등급 변화와 배출집약도 변화 ===')
display(recent_intensity_summary.round(1))

print('\n=== 5. 환경등급 변화폭 vs 배출집약도 변화 ===')

print('전체 기간')
print('상관계수:', round(corr_all, 3))
print('p값:', round(p_all, 4))

print('\n2023→2025')
print('상관계수:', round(corr_recent, 3))
print('p값:', round(p_recent, 4))

=== 1. 환경등급별 배출집약도 ===


,기업연도,평균집약도,중앙값집약도
환경,,,
A+,46,110.5,38.4
A,331,329.5,59.7
B+,304,580.8,87.9
B,243,446.9,125.1
C,391,383.7,171.6
D,51,287.1,151.9



=== 2. 업종별 상관관계 ===
분석 업종 수: 9
음(-)의 상관 업종: 7
양(+)의 상관 업종: 2

=== 3. 최근 환경등급 변화와 절대배출량 변화 ===


,기업연도수,고유기업수,배출량증감률_중앙값,배출감소기업수,배출감소비율
등급변화구분,,,,,
상향,73,69,-0.2,38,52.1
유지,334,210,-0.6,180,53.9
하향,51,50,-1.5,30,58.8



=== 4. 최근 환경등급 변화와 배출집약도 변화 ===


,기업연도수,고유기업수,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
등급변화구분,,,,,
상향,71,67,-4.0,46,64.8
유지,324,205,-2.3,194,59.9
하향,50,49,-2.0,30,60.0



=== 5. 환경등급 변화폭 vs 배출집약도 변화 ===
전체 기간
상관계수: 0.164
p값: 0.0

2023→2025
상관계수: -0.087
p값: 0.0652


In [76]:
# 최종 해석에 사용할 실제 계산값 가져오기

neg_industry = (industry_corr['상관계수'] < 0).sum()
pos_industry = (industry_corr['상관계수'] > 0).sum()
total_industry = len(industry_corr)

a_plus_median = intensity_summary.loc['A+', '중앙값집약도']
a_median = intensity_summary.loc['A', '중앙값집약도']
b_plus_median = intensity_summary.loc['B+', '중앙값집약도']
b_median = intensity_summary.loc['B', '중앙값집약도']
c_median = intensity_summary.loc['C', '중앙값집약도']
d_median = intensity_summary.loc['D', '중앙값집약도']

print(f"""
### KCGS 환경등급과 실제 온실가스 성과 분석

KCGS 환경등급과 GIR 온실가스 배출량, DART 매출액 데이터를 기업과 연도 기준으로 결합해
환경등급과 실제 온실가스 성과의 관계를 확인했다.

먼저 절대 온실가스 배출량만 비교했을 때는 높은 환경등급 기업의 배출량이 오히려 크게 나타나는 경우가 있었다.
하지만 절대배출량은 기업 규모의 영향을 크게 받기 때문에 매출 10억원당 온실가스 배출량인 배출집약도를 추가로 계산했다.

환경등급별 배출집약도 중앙값은 다음과 같다.

- A+ : {a_plus_median:,.1f}
- A  : {a_median:,.1f}
- B+ : {b_plus_median:,.1f}
- B  : {b_median:,.1f}
- C  : {c_median:,.1f}
- D  : {d_median:,.1f}

절대배출량과 달리 기업 규모를 매출액으로 보정한 배출집약도에서는
환경등급이 높은 기업에서 상대적으로 낮은 값이 나타나는 흐름을 확인했다.

다만 업종에 따라 배출 구조 자체가 크게 다르기 때문에 업종별로 다시 비교했다.
분석 가능한 {total_industry}개 업종 가운데
음(-)의 상관관계가 나타난 업종은 {neg_industry}개,
양(+)의 상관관계가 나타난 업종은 {pos_industry}개였다.

또한 같은 기업의 환경등급 변화와 배출집약도 변화를 비교한 결과,
전체 기간의 Spearman 상관계수는 {corr_all:.3f},
2023→2025 구간은 {corr_recent:.3f}으로 나타났다.

이 결과는 환경등급과 온실가스 성과 사이에 일정한 관계가 나타날 수 있음을 보여주지만,
환경등급은 온실가스 배출량만으로 결정되는 지표가 아니며 업종, 기업 규모,
환경관리체계 등 다양한 요소가 함께 반영된다.

따라서 환경등급이 높다는 이유만으로 실제 배출량이 반드시 적다고 볼 수 없고,
기업의 환경성과를 확인할 때는 ESG 평가등급과 함께 절대배출량,
매출 대비 배출집약도, 배출량 변화 추이를 같이 확인할 필요가 있다.
""")


### KCGS 환경등급과 실제 온실가스 성과 분석

KCGS 환경등급과 GIR 온실가스 배출량, DART 매출액 데이터를 기업과 연도 기준으로 결합해
환경등급과 실제 온실가스 성과의 관계를 확인했다.

먼저 절대 온실가스 배출량만 비교했을 때는 높은 환경등급 기업의 배출량이 오히려 크게 나타나는 경우가 있었다.
하지만 절대배출량은 기업 규모의 영향을 크게 받기 때문에 매출 10억원당 온실가스 배출량인 배출집약도를 추가로 계산했다.

환경등급별 배출집약도 중앙값은 다음과 같다.

- A+ : 38.4
- A  : 59.7
- B+ : 87.9
- B  : 125.1
- C  : 171.6
- D  : 151.9

절대배출량과 달리 기업 규모를 매출액으로 보정한 배출집약도에서는
환경등급이 높은 기업에서 상대적으로 낮은 값이 나타나는 흐름을 확인했다.

다만 업종에 따라 배출 구조 자체가 크게 다르기 때문에 업종별로 다시 비교했다.
분석 가능한 9개 업종 가운데
음(-)의 상관관계가 나타난 업종은 7개,
양(+)의 상관관계가 나타난 업종은 2개였다.

또한 같은 기업의 환경등급 변화와 배출집약도 변화를 비교한 결과,
전체 기간의 Spearman 상관계수는 0.164,
2023→2025 구간은 -0.087으로 나타났다.

이 결과는 환경등급과 온실가스 성과 사이에 일정한 관계가 나타날 수 있음을 보여주지만,
환경등급은 온실가스 배출량만으로 결정되는 지표가 아니며 업종, 기업 규모,
환경관리체계 등 다양한 요소가 함께 반영된다.

따라서 환경등급이 높다는 이유만으로 실제 배출량이 반드시 적다고 볼 수 없고,
기업의 환경성과를 확인할 때는 ESG 평가등급과 함께 절대배출량,
매출 대비 배출집약도, 배출량 변화 추이를 같이 확인할 필요가 있다.



In [77]:
print(f"""
### 글로벌 기후목표와 한국 기업의 ESG 환경성과 종합 분석

SBTi 데이터를 통해 글로벌 기업의 기후목표 설정 현황을 먼저 확인했다. 전체 기업 중 76.6%가 단기 감축목표를 설정한 상태였지만, Net-zero 목표가 설정된 기업은 17.9%로 차이가 크게 나타났다. 즉 글로벌 기업의 기후대응은 장기적인 Net-zero 선언보다 단기 감축목표 설정이 더 널리 확산된 상태였다.

한국 기업은 SBTi 데이터에서 125개 기업이 확인됐으며 단기 목표 설정 비율은 72.0%, Net-zero 목표 설정 비율은 14.4%였다. 글로벌 전체보다 다소 낮게 나타났지만 한국 기업은 자동차, 금융, 반도체 등 특정 업종의 비중이 높았고, 업종 구성을 고려한 단순 표준화 결과 단기 목표 설정률 차이는 약 3.1%p 수준으로 줄어들었다. 따라서 단순 국가 간 비율만으로 한국 기업의 기후대응 수준을 판단하기는 어려웠다.

국내에서는 KCGS 환경등급을 통해 ESG 환경평가의 변화를 확인했다. 2021→2022년에는 동일 기업 중 43.8%가 환경등급 하향을 경험했고, 2022→2023년에는 반대로 52.8%가 상향됐다. 이후 2023→2024년과 2024→2025년에는 등급 유지 비율이 각각 70.1%, 75.6%로 높아졌다. 이 때문에 환경등급 변화 자체를 곧바로 실제 환경성과의 개선이나 악화로 해석하지 않고 GIR 배출량 데이터와 추가로 비교했다.

GIR에서 2020~2025년 모두 관측된 동일 기업을 비교한 결과 총 온실가스 배출량은 9.4% 감소했다. 그러나 개별 기업으로 보면 배출량 감소 기업은 51.2%, 증가 기업은 48.6%로 거의 비슷했다. 또한 배출량 감소폭 상위 10개 기업이 전체 감소량의 69.8%를 차지해 전체 감축이 모든 기업에서 고르게 나타난 것은 아니었다.

이후 KCGS 환경등급, GIR 온실가스 배출량, DART 매출액을 결합해 기업 규모를 고려한 배출집약도를 분석했다. 절대배출량에서는 높은 환경등급 기업의 배출량이 오히려 크게 나타나는 경우가 있었지만, 매출액 대비 배출집약도로 비교하면 높은 환경등급 기업에서 상대적으로 낮은 값이 나타나는 흐름을 확인했다.

전체 기간에서 환경등급 변화와 배출집약도 변화의 Spearman 상관계수는 {corr_all:.3f}, 2023→2025년 구간에서는 {corr_recent:.3f}으로 나타났다. 또한 업종별 분석에서도 {len(industry_corr)}개 분석 가능 업종 중 {(industry_corr['상관계수'] < 0).sum()}개 업종에서 환경등급과 배출집약도 사이에 음(-)의 관계가 나타났다.

따라서 이번 분석에서는 ESG 환경등급과 실제 온실가스 성과 사이에 일정한 관계가 확인됐지만, 환경등급만으로 기업의 실제 환경성과를 판단하기에는 한계가 있었다. 기업 규모와 업종에 따라 절대배출량의 차이가 크고, 환경등급 또한 온실가스 배출량 하나만을 평가하는 지표가 아니기 때문이다.

결국 기업의 환경성과를 평가할 때는 ESG 환경등급만 확인하기보다 절대 온실가스 배출량, 매출 대비 배출집약도, 연도별 감축 추이를 함께 관리하는 것이 필요하다고 판단했다.
""")


### 글로벌 기후목표와 한국 기업의 ESG 환경성과 종합 분석

SBTi 데이터를 통해 글로벌 기업의 기후목표 설정 현황을 먼저 확인했다. 전체 기업 중 76.6%가 단기 감축목표를 설정한 상태였지만, Net-zero 목표가 설정된 기업은 17.9%로 차이가 크게 나타났다. 즉 글로벌 기업의 기후대응은 장기적인 Net-zero 선언보다 단기 감축목표 설정이 더 널리 확산된 상태였다.

한국 기업은 SBTi 데이터에서 125개 기업이 확인됐으며 단기 목표 설정 비율은 72.0%, Net-zero 목표 설정 비율은 14.4%였다. 글로벌 전체보다 다소 낮게 나타났지만 한국 기업은 자동차, 금융, 반도체 등 특정 업종의 비중이 높았고, 업종 구성을 고려한 단순 표준화 결과 단기 목표 설정률 차이는 약 3.1%p 수준으로 줄어들었다. 따라서 단순 국가 간 비율만으로 한국 기업의 기후대응 수준을 판단하기는 어려웠다.

국내에서는 KCGS 환경등급을 통해 ESG 환경평가의 변화를 확인했다. 2021→2022년에는 동일 기업 중 43.8%가 환경등급 하향을 경험했고, 2022→2023년에는 반대로 52.8%가 상향됐다. 이후 2023→2024년과 2024→2025년에는 등급 유지 비율이 각각 70.1%, 75.6%로 높아졌다. 이 때문에 환경등급 변화 자체를 곧바로 실제 환경성과의 개선이나 악화로 해석하지 않고 GIR 배출량 데이터와 추가로 비교했다.

GIR에서 2020~2025년 모두 관측된 동일 기업을 비교한 결과 총 온실가스 배출량은 9.4% 감소했다. 그러나 개별 기업으로 보면 배출량 감소 기업은 51.2%, 증가 기업은 48.6%로 거의 비슷했다. 또한 배출량 감소폭 상위 10개 기업이 전체 감소량의 69.8%를 차지해 전체 감축이 모든 기업에서 고르게 나타난 것은 아니었다.

이후 KCGS 환경등급, GIR 온실가스 배출량, DART 매출액을 결합해 기업 규모를 고려한 배출집약도를 분석했다. 절대배출량에서는 높은 환경등급 기업의 배출량이 오히려 크게 나타나

### 분석을 통해 확인한 문제점과 개선 방향

이번 분석에서 가장 크게 확인한 점은 ESG 환경등급 하나만으로 기업의 실제 환경성과를 판단하기 어렵다는 것이다.

첫째, 절대 온실가스 배출량은 기업 규모와 업종의 영향을 크게 받았다. 실제로 환경등급이 높은 기업에서도 절대배출량이 크게 나타나는 경우가 있었다. 따라서 배출량 자체만 비교하기보다 매출액 대비 배출집약도를 함께 확인할 필요가 있다.

둘째, 같은 환경등급이라도 업종에 따라 배출 구조가 달랐다. 발전·철강·화학과 같이 사업 특성상 배출량이 큰 업종과 상대적으로 배출량이 작은 업종을 같은 기준으로 단순 비교하면 기업의 환경성과를 제대로 판단하기 어렵다. 따라서 업종 내 비교가 함께 이루어져야 한다.

셋째, 환경등급의 변화와 실제 온실가스 감축이 항상 같은 방향으로 움직이는 것은 아니었다. 특히 KCGS 환경등급은 2022~2023년에 큰 변동이 나타났기 때문에 단순히 등급이 상승하거나 하락했다는 사실만으로 실제 환경성과의 개선이나 악화를 판단하기 어려웠다.

넷째, GIR 동일 기업 분석에서는 전체 배출량이 감소했지만 감소 규모가 일부 대형 배출기업에 집중되어 있었다. 전체 배출량 감소만 확인하면 많은 기업이 고르게 감축한 것처럼 보일 수 있기 때문에 기업별 증감 여부와 감축 기여도를 함께 확인할 필요가 있다.

따라서 기업의 ESG 환경성과를 관리할 때는 다음과 같이 여러 지표를 함께 관리하는 방식이 필요하다고 판단했다.

- ESG 환경등급
- 절대 온실가스 배출량
- 매출 대비 온실가스 배출집약도
- 전년 대비 배출량 및 배출집약도 변화
- 동일 업종 내 상대적인 수준
- SBTi와 같은 외부 감축목표 설정 여부

특히 환경등급이 상승했지만 배출집약도가 증가하거나, 감축목표를 설정했지만 실제 배출량이 지속적으로 증가하는 기업은 별도로 확인할 수 있어야 한다.

결국 ESG 데이터 관리는 단순히 평가등급이나 목표 설정 여부를 기록하는 것에서 끝나는 것이 아니라, 목표와 실제 성과 데이터를 연결하고 그 차이를 지속적으로 확인하는 방향으로 이루어질 필요가 있다.

In [79]:
# 최종 분석용 데이터 저장

intensity_df.to_csv(
    '../data/esg_environment_final.csv',
    index=False,
    encoding='utf-8-sig'
)

print('저장 완료:', intensity_df.shape)

저장 완료: (1366, 35)
